In [2]:
import regex as re

class RegexTokenizer:
    def __init__(self):
        self.vocab = {}
        self.merges = {}
        self.reverse_vocab = {}
        self.pattern = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
        self.compiled_pattern = re.compile(self.pattern)
        
        
    def get_stats(self, ids, counts = None):
        counts = {} if counts is None else counts
        for pair in zip(ids, ids[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        
        return counts

    def merge_pair(self, ids, pair, new_idx):
        new_ids = []
        i = 0
        
        while i<len(ids):
            
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1]==pair[1]:
                new_ids.append(new_idx)
                i+=2
            
            else:
                new_ids.append(ids[i])
                i+=1
        
        return new_ids

    def train(self, text, vocab_size, verbose=False):
        
        assert vocab_size >= 256, "vocab_size must be at least 256 to accommodate the initial byte tokens"
        number_of_merges = vocab_size - 256
        
        pattern = self.compiled_pattern
        
        token_chunks = pattern.findall(text) # use the regex pattern to split the text into tokens, which will be the initial tokens for the BPE algorithm
        # print(token_chunks)
        
        ids = [list(chunk.encode("utf-8")) for chunk in token_chunks] # convert each token chunk to bytes and then to a list of integers for processing
        
        # print(ids)
        
        # merges = {} # (int, int) -> int
        
        
        # initialize the reverse vocab with the original 256 tokens, where each token is just a single byte corresponding to its index e.g. reverse_vocab[104] = b'h' so in the dictionary {104 : b'h'} and its technically a reverse vocab since it maps from token index to token bytes, but we can call it reverse_vocab for simplicity
        self.reverse_vocab = { idx : bytes([idx]) for idx in range(256)} # idx -> bytes 
        
        for i in range(number_of_merges):
            # count the number of times every consecutive pair appears
            stats = {}
            
            for chunk_ids in ids:
                self.get_stats(chunk_ids, counts=stats) # get the stats for this chunk and add it to the overall stats for this iteration
                
            if not stats:
                if verbose:
                    print("No more pairs to merge, stopping early.")
                break # no more pairs to merge
            
            # find the pair with the highest count
            pair = max(stats, key = stats.get)
                
            idx = 256 + i
            
            # replace all occurrences of pair in ids with idx    
            ids = [self.merge_pair(chunk_ids, pair, idx) for chunk_ids in ids] # merge the pair in each chunk
            
            self.merges[pair] = idx
            self.reverse_vocab[idx] = self.reverse_vocab[pair[0]] + self.reverse_vocab[pair[1]] # new token is concatenation of merged tokens
            
            if verbose:
                print(f"merge {i+1}/{number_of_merges}: {pair} -> {idx} ({self.reverse_vocab[idx]}) had {stats[pair]} occurrences")

        
        self.vocab = {v : k for k,v in self.reverse_vocab.items()} # not needed for encoding since we can just use the merges to merge pairs in the input tokens, but could be useful for debugging or other purposes and we dont use it in this implementation since we can just use the merges to merge pairs in the input tokens, but could be useful for debugging or other purposes 
        
    def encode_chunks(self, chunk):
        
        chunk_ids = list(map(int, chunk)) # convert the chunk to a list of integers for processing
        
        while len(chunk_ids) >= 2:
                stats = self.get_stats(chunk_ids)
                pair = min(stats, key = lambda p : self.merges.get(p, float('inf'))) # find the pair with the smallest new index in merges that still exists in ids because we need to apply merges in the same order they were created for correct encoding
            
                if pair not in self.merges:
                    break # no more pairs to merge, we’re done
                
                # print(pair)
                
                chunk_ids = self.merge_pair(chunk_ids, pair, self.merges[pair]) # merge the pair in the ids list
        
        return chunk_ids

    def encode(self, text):
        
        token_chunks = self.compiled_pattern.findall(text) # use the regex pattern to split the text into tokens, which will be the initial tokens for the BPE algorithm
        
        ids = [] # all chunks of text are encoded separately, then results are joined
        
        for token_chunk in token_chunks:
            chunk_bytes = token_chunk.encode("utf-8") # convert the chunk to bytes
            chunk_ids = self.encode_chunks(chunk_bytes) # encode the chunk using the same merges that were learned during training, which will give us a list of token ids for this chunk
            ids.extend(chunk_ids) # add the token ids for this chunk to the overall list of token ids for the whole input text
        
        return ids
    
    def decode(self, ids):
        text = bytes().join(self.reverse_vocab[idx] for idx in ids) # get the byte sequence for each token index and concatenate them, using an empty byte string for any unknown indices (shouldn’t happen if encoding was done with the same vocab)
        return text.decode("utf-8", errors="replace") # decode the bytes back to a string, replacing any invalid sequences with the Unicode replacement character
    

In [7]:
regexTokenizer = RegexTokenizer()

with open("TinyStoriesV2-GPT4-train.txt", "r", encoding="utf-8") as f:
    text = f.read(50_000_000) # read the first 50 million characters for training, which should be enough to learn a good BPE vocab without taking too long to train    



In [9]:
regexTokenizer.train(text, vocab_size=2000, verbose=True)

merge 1/1744: (32, 116) -> 256 (b' t') had 1426225 occurrences
merge 2/1744: (104, 101) -> 257 (b'he') had 1420332 occurrences
merge 3/1744: (32, 97) -> 258 (b' a') had 1065420 occurrences
merge 4/1744: (32, 115) -> 259 (b' s') had 727585 occurrences
merge 5/1744: (110, 100) -> 260 (b'nd') had 710958 occurrences
merge 6/1744: (32, 119) -> 261 (b' w') had 707611 occurrences
merge 7/1744: (256, 257) -> 262 (b' the') had 649894 occurrences
merge 8/1744: (101, 100) -> 263 (b'ed') had 557483 occurrences
merge 9/1744: (32, 98) -> 264 (b' b') had 496454 occurrences
merge 10/1744: (256, 111) -> 265 (b' to') had 468346 occurrences
merge 11/1744: (258, 260) -> 266 (b' and') had 437362 occurrences
merge 12/1744: (32, 104) -> 267 (b' h') had 399893 occurrences
merge 13/1744: (32, 102) -> 268 (b' f') had 381561 occurrences
merge 14/1744: (105, 110) -> 269 (b'in') had 372425 occurrences
merge 15/1744: (32, 84) -> 270 (b' T') had 369292 occurrences
merge 16/1744: (261, 97) -> 271 (b' wa') had 362608 

In [10]:
model_file = "bpe_tokenizer" + ".model"
with open(model_file, "w") as f:
    f.write("bpe_tokenizer v1\n")
    f.write(f"{regexTokenizer.pattern}\n")

# implement when we have special tokens   
    # f.write(f"{len(regexTokenizer.special_tokens)}\n")
    # for special, idx in regexTokenizer.special_tokens.items():
    #     f.write(f"{special} {idx}\n")
    
    for idx1, idx2 in regexTokenizer.merges:
        f.write(f"{idx1} {idx2}\n")


In [11]:
vocab_file = "bpe_tokenizer" + ".vocab"
inverted_merges = {v:k for k,v in regexTokenizer.merges.items()} # idx -> (idx1, idx2)
with open(vocab_file, "w", encoding = "utf-8") as f:
    for idx, token in regexTokenizer.reverse_vocab.items():
        s = regexTokenizer.decode([idx]) # decode the token bytes back to a string for saving in the vocab file, which will be useful for debugging and understanding what each token index corresponds to in terms of text
        if(idx in inverted_merges):
            idx1, idx2 = inverted_merges[idx]
            s1 = regexTokenizer.decode([idx1])
            s2 = regexTokenizer.decode([idx2])
            f.write(f"[{s1}][{s2}] -> [{s}] {idx}\n")
        else:
            f.write(f"[{s}] {idx}\n")

In [12]:
text = "hello hello hello"

encoded = regexTokenizer.encode(text)  # fix encode and decode for regex logic
decoded = regexTokenizer.decode(encoded)

print(encoded)
print(encoded == regexTokenizer.encode(text)) # should be True
print(len(encoded))
print(decoded)

[257, 294, 111, 285, 294, 111, 285, 294, 111]
True
9
hello hello hello


In [13]:
text = "hello world!!!? (안녕하세요!) lol123 😉"

encoded = regexTokenizer.encode(text)
decoded = regexTokenizer.decode(encoded)

print(encoded)
print(encoded == regexTokenizer.encode(text)) # should be True
print(len(encoded))
print(decoded)

[257, 294, 111, 1577, 33, 33, 33, 63, 32, 40, 236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 148, 33, 41, 444, 108, 49, 50, 51, 32, 240, 159, 152, 137]
True
37
hello world!!!? (안녕하세요!) lol123 😉


In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
print(ids)
print(len(ids))
print(enc.decode(ids))
print(enc.n_vocab)